In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    collist,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    TransplantPostOPID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    EmpfaengerID,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
    fix_units,
)

### Target Population Filtering

The patients in the dataset were filtered to match the target population (see [](general:tpf)). Rows contributed from the {term}`ET` foundation include the identifier of the recipient and the transplant identifie, while rows from the {term}`IQTIG` only include a recipient identifier. Therfore, {term}`ET` rows can be filtered based on the transplantations in the targetpopulation, while {term}`IQTIG` can only be filtered based on the recipient.

Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
split_data(
    data,
    [
        "transplant_et_id",
        "recipient_et_id_et",
        "recipient_et_iqtig",
    ],
)
data["recipient_id"] = collapse_col(
    data.loc[:, ["recipient_et_id_et", "recipient_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
olres = data["recipient_id"].drop_duplicates()
assert not data["recipient_id"].isna().any(), "Missing recipient info"
assert (
    data["date_et"] - data["date_iqtig"]
).dropna().abs().max() == 0, "Date disagreements"
targetpop = pd.read_parquet(targetpop_data)
assert (
    not targetpop["recipient_et_id_et"].duplicated().any()
), "Repeat recipients currently can not be matched due to IQTIG data"

data = (
    pd.merge(
        data,
        targetpop.loc[:, ["recipient_et_id_et", "transplant_et_id"]],
        left_on="recipient_id",
        right_on="recipient_et_id_et",
    )
    .drop(columns="recipient_et_id_et_y")
    .rename(columns={"recipient_et_id_et_x": "recipient_et_id_et"})
)
data = (
    data[
        (data["transplant_et_id_y"] == data["transplant_et_id_x"])
        | data["transplant_et_id_x"].isna()
    ]
    .drop(columns="transplant_et_id_y")
    .rename(columns={"transplant_et_id_x": "transplant_et_id"})
    .drop_duplicates()
)

display(
    Markdown(
        f"""The filter process reduces the number of recipients in the data ({olres.size}) and target population ({targetpop["recipient_et_id_et"].nunique()})
            to {data["recipient_id"].nunique()} in the processed data.
        """
    )
)
del targetpop, olres

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file the {term}`IQTIG` and {term}`ET` data is partly already connected (see [](general:ic)). Rows from the different institutes were joined when they had the same date and most likely refer to the same follow-up visit.

In [ ]:
idcols = [
    "transplant_et_id",
    "recipient_et_id_et",
    "recipient_et_iqtig",
]
assert len(split_data(data, idcols)) == 3, "Not 3 different row types present!?"
del idcols

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

While there is no column differentiating between different types of follow-up types (An exception is the {term}`IQTIG` which marks the first and the recurrent visits), we include an indicator for the following analysis based on which data contributor includes a date on the respective lines (see [](general:rf)). The column `recipient_id` was constructed by consolidating `recipient_et_id_et` + `recipient_et_iqtig` and the column `date` from `date_et` + `date_iqtig`.

In [ ]:
data["Institute with a follow-up date"] = (
    (~data["date_et"].isna()) + (~data["date_iqtig"].isna()) * 2
).replace({1: "ET", 2: "IQTIG", 3: "ET+IQTIG", 0: "No Date"})
data["date"] = collapse_col(
    data.loc[:, ["date_et", "date_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "recipient_id",
    ],
    "date",
    "Institute with a follow-up date",
)
data.drop(
    columns=["Institute with a follow-up date", "date", "recipient_id"], inplace=True
)

### Unit Conversions

We applied common translations and converted the the creatinine measurements to a common unit (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
fix_units(
    data,
    "discharge_serum_creatinine_mg_per_dl",
    "discharge_serum_creatinine_unit",
    config["data"]["unit_conversions"]["creatinine"]["target"],
    config["data"]["unit_conversions"]["creatinine"]["factors"],
)

In [ ]:
cols = data.columns[data.columns.to_series().str.endswith("unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1]
data = data.drop(columns=dropme)
display(
    Markdown(
        f"The columns {collist(dropme)} were removed as only a single unit was used."
    )
)

### Consolidating Columns

We consolidated columns that appear for both {term}`ET` and {term}`IQTIG` (see [](general:crc)).

In [ ]:
red = find_redundant_cols(data)
red["recipient_et_id_et"] = ["recipient_et_id_et", "recipient_et_iqtig"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `date` column as the time axis.

In [ ]:
indcols = ["recipient_et_id_et", "transplant_et_id"]
data = data.sort_index(axis=1).sort_values(indcols + ["date"], axis=0)
data = data.set_index(indcols)


assert (
    data["cold_ischemia_time_liver_min"]
    .groupby(data.index.names)
    .apply(lambda vals: vals.std())
    .dropna()
    .shape[0]
    == 0
)
assert (
    data["transplantation_duration_min"]
    .groupby(data.index.names)
    .apply(lambda vals: vals.std())
    .dropna()
    .shape[0]
    == 0
)
assert (
    data["warm_ischemia_time_second_min"]
    .groupby(data.index.names)
    .apply(lambda vals: vals.std())
    .dropna()
    .shape[0]
    == 0
)


data.drop(
    columns=[
        "cold_ischemia_time_liver_min",
        "transplantation_duration_min",
        "warm_ischemia_time_second_min",
    ],
    inplace=True,
)

In [ ]:
question = ["no", "yes", "unknown"]


class FollowupNiere(TransplantPostOPID, EmpfaengerID):
    acute_rejection: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Acute Rejection",
        description="Was an acute rejection diagnosed?",
        isin=question,
    )
    acute_rejection_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Acute Rejection Date",
        description="When was the acute rejection diagnosed?",
    )
    acute_rejections_count: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Acute Rejection Count",
        description="How many acute rejections were diagnosed?",
        ge=0,
    )
    arterial_clot: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Arterial Clot",
        description="Was an arterial clot diagnosed?",
        isin=question,
    )
    artery_thrombosis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Arterial Thrombosis",
        description="Was an arterial thrombosis diagnosed?",
        isin=question,
    )
    bleeding: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bleeding",
        description="Did the graft bleed?",
        isin=question,
    )
    blind_trial: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Blind Trial",
        description="Was the patient part of a blind trail?",
        isin=question,
    )
    cerebrovascular_problem: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cerebrovascular Complications",
        description="Were cerebrovascular complications diagnosed?",
        isin=question,
    )
    changed_immunosuppression: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Changes in Immunosuppression Medication",
        description="Was the immunosuppression medication changed?",
        isin=question,
    )
    changed_immunosuppression_reason: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Reason For Changes in Immunosuppression Medication",
        description="Why was the immunosuppression medication changed?",
        isin=["Other", "Acute rejection", "Chronic Rejection", "Intolerance"],
    )
    chronic_hepatitis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Chronic Hepatitis",
        description="Was chronic hepatitis diagnosed?",
        isin=question,
    )
    chronic_obstructive_lung_disease: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Chronic Obstructive Lung Disease",
        description="Was chronic obstructive lung disease diagnosed?",
        isin=question,
    )
    chronic_rejection: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Chronic Rejection",
        description="Was chronic rejection diagnosed?",
        isin=question,
    )
    coronary_artery_disease: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Coronary Artery Disease",
        description="Was Coronary artery disease diagnosed?",
        isin=question,
    )
    creatinine_umol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Creatinine",
        description="Creatinine measurement in umol/l",
        ge=0,
    )
    creatinine_unknown: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Creatinine Unknown",
        description="Was the creatinine marked as unknown?",
        isin=["yes"],
    )
    date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Follow-Up Date",
        description="When was the follow-up conducted?",
    )
    death_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Follow-Up Death Date",
        description="When did the patient die? Added during follow-up.",
    )
    death_reason: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Follow-Up Death Reason",
        description="Why did the patient die? Added during follow-up.",
        isin=[
            "cerebrovaskulär",
            "kardiovaskulär",
            "Infektion",
            "Malignom",
            "andere",
            "unknown",
        ],
    )
    dehiscence: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dehiscence",
        description="Was wound dehiscence diagnosed?",
        isin=question,
    )
    diabetes: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Diabetes",
        description="Was diabetes diagnosed?",
        isin=question,
    )
    dialysis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dialysis",
        description="Was dialysis diagnosed?",
        isin=question,
    )
    dialysis_after_operation_count: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dialysis Count",
        description="How many dialysis were performed after the operation?",
    )
    discharge_serum_creatinine_mg_per_dl: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Serum creatinine at Discharge",
        description="What was the creatinine measurement at discharge in blood serum in mg/dl?",
        ge=0,
    )
    discharge_weight_kg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Body Weight at Discharge",
        description="How much did the patient weigh in kg at discharge?",
        ge=0,
    )
    duration_years: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="IQTIG Follow-Up year",
        description="How many years was the follow-up by IQTIG?",
        isin=[1, 2, 3, 4],
    )
    followup_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="IQTIG Follow-Up Visit Type",
        description="Was it a first follow-up visit?",
        isin=["Initial", "Recurrent"],
    )
    graft_failure: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Graft Failure",
        description="Was graft failure diagnosed?",
        isin=question,
    )
    graft_failure_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Graft Failure Date",
        description="When was graft failure diagnosed?",
    )
    graft_failure_reason: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Graft Failure Reason",
        description="Why did the graft fail?",
        isin=[
            "andere",
            "Rejektion",
            "primäre Nichtfunktion",
            "Blutung",
            "Infektion im OP-Bereich",
            "Gefäßverschluss",
            "Rekurrenz der Grunderkrankung",
            "De Novo Nierenerkrankung",
        ],
    )
    graft_function: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Graft Function",
        description="Is the graft functional?",
        isin=["yes", "no"],
    )
    graft_function_delayed: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Graft Function Delayed",
        description="Was the graft function delayed?",
        isin=[
            "Kidney functions directly after transplant",
            "Not direct",
            "Kidney functions sometime after transplant",
            "Kidney functions never after transplant",
            "unknown",
        ],
    )
    height_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Patient Height",
        description="What is the height of the patient in cm?",
    )
    hospitilization_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hospitilization Date",
        description="When was the patient hospitilizated?",
    )
    hypertension: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hypertension",
        description="Was hypertension diagnosed?",
        isin=question,
    )
    induction_therapy: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Induction Therapy",
        description="Was the patient treated with induction therapy?",
        isin=question,
    )
    last_dialysis_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Last Dialysis Date",
        description="When was the last dialysis performed?",
    )
    malgin: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Malgin Tumor",
        description="Was a malgin tumor diagnosed?",
        isin=["Lymphoma", "Skin Tumor", "Carcinoma"],
    )
    other_complications: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Other Complications",
        description="Were there other complications?",
        isin=question,
    )
    other_diseases: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Other Diseases",
        description="Were other diseases diagnosed?",
        isin=question,
    )
    other_diseases: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Other Transplant Diseases",
        description="Were other transplantation diseases diagnosed?",
        isin=question,
    )
    patient_died: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Patient Death",
        description="Did the patient die?",
        isin=question,
    )
    rejection: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Organ rejection",
        description="Was the organ rejected?",
        isin=question,
    )
    rejections_count: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Organ Rejection Count",
        description="How many rejections were observed?",
    )
    survey_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Survey Type",
        description="How was the patient surveyed?",
    )
    treated_chronic_rejection: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Chronic Rejection Treatment",
        description="Was chronic rejected treated?",
        isin=question,
    )
    treated_chronic_rejections_count: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Chronic Rejection Treatment Count",
        description="How many times was chronic rejection treated?",
    )
    trial_immunosuppressants: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Trial of Immunosuppressants",
        description="Was the patient part of a trial testing immunosuppressants?",
        isin=question,
    )
    urine_leakage: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Urine Leakage",
        description="Was urine leakage diagnosed?",
        isin=question,
    )
    vein_thrombosis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Vein Thrombosis",
        description="Was a vein thrombosis diagnosed?",
        isin=question,
    )
    weight_kg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Body Weight",
        description="How much did the patient weigh in kg?",
        ge=0,
    )
    work_ability: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Work Ability",
        description="Was the patient able to work?",
        isin=question,
    )
    wound_infection_present: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Wound Infection",
        description="Was a wound infection present?",
        isin=question,
    )
    wound_infection_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Wound Infection Type",
        description="Where was the infection?",
        isin=["Deep incisional", "Organ/Space", "Superficial"],
    )

    class Config:
        title = "Follow-Up Kidney Dataset"
        description = "Each row represents a performed follow-up by either ET, IQTIG or both. The data is based on the 'element_followup_niere.csv' file."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(FollowupNiere, data)

In [ ]:
FollowupNiere.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    FollowupNiere.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)